# QT - Quantum Turbulence in a Dipolar Supersolid

Validated eGPE split-step solver.  **`Runtime -> Run all`.**  Nothing to edit.

Outputs go to `MyDrive/Research/Quantum_Turbulence/<date>_<time>_<title>/`
with `checkpoints/ diagnostics/ figures/ logs/`.  Run time 8-12 minutes.

**Version 1.1** fixes a temporal-aliasing blow-up present in v1.0:
the demo used a timestep at the aliasing resonance and diverged near
t=35 with the norm still conserved.  The solver now rejects such
timesteps, and five regression checks (T7) pin the fix down.

## 1. Setup - clone, import, mount Drive

In [ ]:
# CELL 1 - Setup.  Run this first.  Nothing to edit.
import os, sys, subprocess

REPO = 'https://github.com/vk001king/qt.git'
DEST = '/content/qt'

if os.path.isdir(DEST + '/.git'):
    print('Repo present; pulling latest...')
    subprocess.run(['git', '-C', DEST, 'pull', '-q'], check=False)
else:
    print('Cloning', REPO)
    r = subprocess.run(['git', 'clone', '-q', REPO, DEST],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone failed:\n' + r.stderr)
print('Repo ready at', DEST)

need = ['__init__.py', 'kernels.py', 'lhy.py', 'solver.py',
        'diagnostics.py', 'drive_io.py']
have = os.listdir(DEST + '/qtsim')
miss = [f for f in need if f not in have]
if miss:
    raise RuntimeError('Repo incomplete, missing: %s' % miss)
print('All 6 package files present')

if DEST in sys.path:
    sys.path.remove(DEST)
sys.path.insert(0, DEST)
for m in [k for k in sys.modules if k.split('.')[0] == 'qtsim']:
    del sys.modules[m]

import qtsim
from qtsim import Grid, EGPESolver, Archive
assert qtsim.selfcheck(), 'qtsim self-check failed'
print('Loaded qtsim v%s from %s'
      % (qtsim.__version__, os.path.dirname(qtsim.__file__)))

# Archive now REFUSES to run if Drive fails to mount, rather than
# silently writing to ephemeral disk (that cost a 200 s run in v1.3).
# If you get a RuntimeError here: approve the Drive popup and re-run.
archive = Archive(base='Research', project='Quantum_Turbulence')

assert archive.project_dir.startswith('/content/drive/MyDrive'), \
    'Archive is not on Google Drive -- outputs would not persist'
print('CONFIRMED persistent:', archive.project_dir)

# If a previous session wrote to ephemeral disk, pull those runs across:
import os
if os.path.isdir('/content/drive_local'):
    from qtsim import rescue_ephemeral
    rescue_ephemeral()

if not archive.persistent:
    raise RuntimeError(
        'Outputs are NOT on Google Drive (root: %s). They would be lost '
        'when this session ends. Re-run this cell and approve the Drive '
        'popup. To proceed anyway: Archive(..., require_drive=False)'
        % archive.root)
print('Outputs are persistent:', archive.project_dir)
archive.list_runs()

## 2. Validation - 23 checks against analytic results

If anything fails, stop: nothing below can be trusted.

In [ ]:
# CELL 2 - Validation suite (23 checks).  About 60 s.
import subprocess, sys

run = archive.new_run('validation suite',
                      params={'version': qtsim.__version__})
r = subprocess.run([sys.executable, 'tests/run_validation.py'],
                   cwd=DEST, capture_output=True, text=True)
print(r.stdout[-4000:])
if r.stderr.strip():
    print('STDERR:', r.stderr[-1500:])

run.save_text(r.stdout, 'validation-output')
if r.returncode == 0:
    run.finish('completed')
    print('RESULT: all checks passed.')
else:
    run.finish('failed')
    print('RESULT: failures above.  Do not trust the physics below.')

## 3. Vortex pair - motion and sound emission

Energy drift is printed alongside the physics as a live health check.

In [ ]:
# CELL 3 - Vortex pair: detection, motion, sound emission.
#
# TIMESTEP NOTE (this bit matters).  The kinetic substep applies
# exp(-i k^2 dt) exactly, so there is no CFL stability limit -- but the
# highest grid mode advances k_max^2 * dt radians per step.  Once that
# approaches pi those modes are unresolved in time and the nonlinear
# term pumps them until the field detonates, WITH THE NORM STILL
# CONSERVED to round-off.  An earlier version of this notebook used
# 256^2 with dt=0.02 (6.3 rad/step) and blew up near t=35: two vortices
# became 21000.  The solver now refuses such timesteps outright.
import numpy as np, matplotlib.pyplot as plt
from qtsim import Grid, EGPESolver
from qtsim import plaquette_charges_2d, helmholtz_split_2d

N, L, dt = 128, 64.0, 0.01
grid = Grid((N, N), (L, L))
dx = grid.dx[0]

run = archive.new_run('vortex pair dynamics',
                      params={'N': N, 'L': L, 'dt': dt})

probe = EGPESolver(grid)
print('grid  dx = %.3f xi   (need dx <= 0.5 to resolve the core)' % dx)
print('phase advance of highest mode = %.3f rad/step'
      % probe.max_phase_per_step(dt))
print('largest safe dt on this grid  = %.4f' % probe.suggested_dt())
print()

# half-cell offsets: a core sitting exactly on a grid point has
# undefined phase there and corrupts the winding detector
xp, xm, yc = 8.0 + 0.5*dx, -8.0 + 0.5*dx, 0.5*dx
thp = np.arctan2(grid.X[1] - yc, grid.X[0] - xp)
thm = np.arctan2(grid.X[1] - yc, grid.X[0] - xm)
rp = np.sqrt((grid.X[0] - xp)**2 + (grid.X[1] - yc)**2)
rm = np.sqrt((grid.X[0] - xm)**2 + (grid.X[1] - yc)**2)

s = EGPESolver(grid)
s.psi = np.tanh(rp) * np.tanh(rm) * np.exp(1j*(thp - thm))
s.step_imag(5e-3, 300, norm_target=s.norm())

q = plaquette_charges_2d(s.psi)
print('vortices detected:  +1 = %d   -1 = %d'
      % (int((q == 1).sum()), int((q == -1).sum())))

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(np.abs(s.psi).T**2, origin='lower',
             extent=[-L/2, L/2, -L/2, L/2], cmap='inferno')
ax[0].set_title('density |psi|^2')
ax[0].set_xlabel('x / xi'); ax[0].set_ylabel('y / xi')
ax[1].imshow(np.angle(s.psi).T, origin='lower',
             extent=[-L/2, L/2, -L/2, L/2], cmap='twilight')
for v in np.argwhere(q == 1):
    ax[1].plot(grid.X[0][v[0], v[1]], grid.X[1][v[0], v[1]],
               'r+', ms=14, mew=2)
for v in np.argwhere(q == -1):
    ax[1].plot(grid.X[0][v[0], v[1]], grid.X[1][v[0], v[1]],
               'bx', ms=14, mew=2)
ax[1].set_title('phase  (red +1, blue -1)')
ax[1].set_xlabel('x / xi')
plt.tight_layout()
run.save_figure(fig, 'pair-initial')
plt.show()

# evolve, watching conservation as a live health check
E0, N0 = s.energy()['total'], s.norm()
ts, Ei_s, Ec_s, nv_s, dE_s = [], [], [], [], []
nsteps = int(100.0 / dt)
print('evolving to t = 100 ...')
print('     t     nv        Ei        Ec      dE/E')
for k in range(nsteps + 1):
    if k % 250 == 0:
        Ei, Ec, _ = helmholtz_split_2d(s.psi, grid)
        nv = int((plaquette_charges_2d(s.psi) != 0).sum())
        dE = abs(s.energy()['total'] - E0) / abs(E0)
        ts.append(k*dt); Ei_s.append(Ei); Ec_s.append(Ec)
        nv_s.append(nv); dE_s.append(dE)
        if k % 2500 == 0:
            print('  %6.1f  %4d  %8.2f  %8.2f  %8.1e'
                  % (k*dt, nv, Ei, Ec, dE))
    if k < nsteps:
        s.step_real(dt)

dE_max = max(dE_s)
print()
print('max energy drift  = %.2e   (healthy if < 1e-5)' % dE_max)
print('norm drift        = %.2e' % (abs(s.norm() - N0)/N0))
print('final vortex count= %d   (started at 2)' % nv_s[-1])
if dE_max > 1e-5 or nv_s[-1] > 4:
    print('WARNING: run is not trustworthy -- treat output as a bug.')
else:
    print('Run is conservative and stable.')

fig2, (a1, a2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
a1.plot(ts, Ei_s, 'b-', label='incompressible (vortex)')
a1.plot(ts, Ec_s, 'r-', label='compressible (sound)')
a1.set_ylabel('energy'); a1.legend()
a1.set_title('energy channels for a travelling vortex pair')
a2.plot(ts, nv_s, 'ko-', ms=3)
a2.set_xlabel('time / tau'); a2.set_ylabel('vortex count')
a2.set_ylim(0, max(6, max(nv_s) + 1))
plt.tight_layout()
run.save_figure(fig2, 'energy-channels')
plt.show()

run.save_diagnostics([{'t': a, 'Ei': b, 'Ec': c, 'nv': d, 'dE_rel': e}
                      for a, b, c, d, e in
                      zip(ts, Ei_s, Ec_s, nv_s, dE_s)])
run.save_checkpoint(s.psi, step=nsteps, t=100.0)
run.finish('completed')

## 4. Quasi-2D dipolar crystal (v1.3)

Uses the projected kernel with a roton. Compares the crystal period
against linear theory and checks real-time stationarity honestly.

In [ ]:
# CELL 4 - Quasi-2D dipolar ground state with the PROJECTED kernel.
#
# v1.3 replaces the bare periodic dipolar symbol with the quasi-2D
# projected kernel derived by integrating the 3D symbol against the
# Gaussian axial density:
#   D(k) = 2*sqrt(2) - 3*sqrt(2*pi)*u*erfcx(u),   u = k*l_z/2
# It runs from +2.828 at k=0 to -1.414 at large k.  That SIGN CHANGE is
# the roton, and it is exactly what the bare kernel lacked -- which is
# why v1.2 produced only a box-scale stripe.  Verified against direct
# quadrature to 5e-15 (test T8a).
import numpy as np, matplotlib.pyplot as plt
from qtsim import Grid, EGPESolver, gamma_tilde, Q5
from qtsim import quasi2d_dipolar_symbol, quasi2d_dipolar_profile
from qtsim.minimize import multistart_ground_state

eps_dd, l_z, n0as3 = 1.8, 6.0, 5e-5
gam = gamma_tilde(eps_dd, n0as3)
run = archive.new_run('quasi2d dipolar crystal',
                      params={'eps_dd': eps_dd, 'l_z': l_z,
                              'n0_as3': n0as3, 'kernel': 'quasi2d'})

# --- linear theory: where does the roton sit? -------------------------
kk = np.linspace(1e-6, 12, 40000)
Dp = quasi2d_dipolar_profile(kk, l_z)
inside = kk**2 + 2.0*(1.0 + eps_dd*Dp) + 3.0*gam
imin = int(np.argmin(inside))
k_rot, a_pred = kk[imin], 2*np.pi/kk[imin]
unstable = inside[imin] < 0
print('eps_dd = %.2f   l_z = %.1f xi   gamma = %.5f' % (eps_dd, l_z, gam))
print('roton k   = %.4f      predicted lattice period a = %.3f xi'
      % (k_rot, a_pred))
print('uniform state unstable: %s' % unstable)
print()

# --- commensurate box for a triangular lattice ------------------------
nx, ny = 5, 6
Lx, Ly = nx*a_pred, ny*a_pred*np.sqrt(3)/2
Nx = int(round(Lx/0.25/2)*2); Ny = int(round(Ly/0.25/2)*2)
grid = Grid((Nx, Ny), (Lx, Ly))
Dk = quasi2d_dipolar_symbol(grid, l_z)
print('commensurate box %.2f x %.2f xi   grid %dx%d' % (Lx, Ly, Nx, Ny))

# v1.6: multi-start L-BFGS-B instead of plain gradient flow.
# Gradient flow plateaued at mu = 5.509 after 24000 steps; L-BFGS from one
# seed reached 5.466; multi-start with an ideal-lattice seed reached 5.189.
# The landscape has many nearby minima, so both a better optimiser and
# several starts matter.
def lattice_seed(grid, a=a_pred, nx=nx, ny=ny, Lx=Lx, Ly=Ly):
    """Ideal triangular droplet lattice as a physically motivated start."""
    b1 = np.array([a, 0.0]); b2 = np.array([a/2, a*np.sqrt(3)/2])
    dens = np.zeros(grid.shape)
    for ii in range(-1, nx+2):
        for jj in range(-1, ny+2):
            cc = ii*b1 + jj*b2
            for sx in (-Lx, 0, Lx):
                for sy in (-Ly, 0, Ly):
                    dens += np.exp(-((grid.X[0]-cc[0]-sx)**2
                                     + (grid.X[1]-cc[1]-sy)**2)/(2*1.2**2))
    return 0.05 + dens

Nt = float(grid.integrate(np.ones(grid.shape)))
print('multi-start L-BFGS-B search ...')
s, opt = multistart_ground_state(grid, eps_dd, gam, Dk, Nt, n_seeds=4,
                                 lattice_seed=lattice_seed,
                                 presmooth=2000, maxiter=2500, verbose=True)
mu, res = opt['mu'], opt['residual']
print('best mu = %.6f   residual = %.3e   (res/mu = %.2e)'
      % (mu, res, res/mu))

n = np.abs(s.psi)**2
contrast = float((n.max()-n.min())/n.mean())
sp = np.abs(np.fft.fftn(n - n.mean())); sp[0,0] = 0.0
j = np.unravel_index(np.argmax(sp), sp.shape)
k_m = float(np.hypot(grid.K[0][j], grid.K[1][j]))
lam_m = 2*np.pi/k_m          # density MODULATION WAVELENGTH
err = abs(lam_m - a_pred)/a_pred*100
# For a triangular lattice the first Bragg vector has |k| = 4*pi/(a*sqrt3),
# so the nearest-neighbour lattice constant is a = 2*lambda/sqrt(3).
# Conflating the two is a factor 1.1547 error -- and published 164Dy
# numbers quote the LATTICE CONSTANT, so this matters for V4.
a_tri_m = 2*lam_m/np.sqrt(3)
a_tri_p = 2*a_pred/np.sqrt(3)
a_m = lam_m                  # keep the old name for downstream code
print()
print('contrast                     = %.3f' % contrast)
print('modulation wavelength lambda = %.4f xi  (predicted %.4f) -> %.2f%%'
      % (lam_m, a_pred, err))
print('triangular lattice constant  = %.4f xi  (predicted %.4f)'
      % (a_tri_m, a_tri_p))
print('  (a = 2*lambda/sqrt(3); do not confuse the two when comparing')
print('   against published 164Dy lattice constants)')

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
im = ax[0].imshow(n.T, origin='lower',
                  extent=[-Lx/2, Lx/2, -Ly/2, Ly/2], cmap='viridis')
plt.colorbar(im, ax=ax[0], label='density n')
ax[0].set_title('droplet crystal, eps_dd=%.1f, l_z=%.0f xi' % (eps_dd, l_z))
ax[0].set_xlabel('x / xi'); ax[0].set_ylabel('y / xi')
sps = np.abs(np.fft.fftshift(np.fft.fftn(n - n.mean())))
kxs = np.fft.fftshift(np.fft.fftfreq(Nx, d=Lx/Nx))*2*np.pi
kys = np.fft.fftshift(np.fft.fftfreq(Ny, d=Ly/Ny))*2*np.pi
ax[1].imshow(sps.T, origin='lower',
             extent=[kxs[0], kxs[-1], kys[0], kys[-1]],
             cmap='magma', vmax=sps.max()*0.3)
ax[1].set_xlim(-3, 3); ax[1].set_ylim(-3, 3)
ax[1].set_title('structure factor |n(k)|')
ax[1].set_xlabel('k_x'); ax[1].set_ylabel('k_y')
plt.tight_layout()
run.save_figure(fig, 'quasi2d-crystal')
plt.show()

# --- HONEST stationarity check, done properly --------------------------
# A pointwise comparison of |psi|^2 flags any translation as "unstable",
# which is wrong for a crystal.  v1.5 measures translation-INVARIANT
# quantities as well, which is what distinguishes the possible causes:
#   contrast + n_peak stable      -> droplets keep shape (no collapse)
#   Bragg phase shift small       -> not a rigid Goldstone slide
#   Bragg AMPLITUDES change       -> droplets rearranging (wrong minimum)
n_before = n.copy()
A_before = np.abs(np.fft.fftn(n_before - n_before.mean()))
c_before = float((n_before.max() - n_before.min())/n_before.mean())
sp0 = np.fft.fftn(n_before - n_before.mean()); sp0[0, 0] = 0.0
jj = np.unravel_index(np.argmax(np.abs(sp0)), sp0.shape)
kvec = float(np.hypot(grid.K[0][jj], grid.K[1][jj]))
ph_before = float(np.angle(sp0[jj]))
E_before, N_before = s.energy()['total'], s.norm()

dt_safe = min(0.0005, s.suggested_dt()*0.8)
s.step_real(dt_safe, int(6.0/dt_safe))

n_after = np.abs(s.psi)**2
A_after = np.abs(np.fft.fftn(n_after - n_after.mean()))
spA = np.fft.fftn(n_after - n_after.mean()); spA[0, 0] = 0.0
dE = abs(s.energy()['total'] - E_before)/abs(E_before)
dN = abs(s.norm() - N_before)/N_before
drift_point = float(np.abs(n_after - n_before).max()/n_before.mean())
dA = float(np.linalg.norm(A_after - A_before)/np.linalg.norm(A_before))
c_after = float((n_after.max() - n_after.min())/n_after.mean())
dcon = abs(c_after - c_before)/c_before
dph = float(np.angle(np.exp(1j*(np.angle(spA[jj]) - ph_before))))
slide = dph/kvec

print()
print('real-time diagnostics over t = 6:')
print('  energy drift            %.2e   (solver health)' % dE)
print('  norm drift              %.2e' % dN)
print('  pointwise |dn|/n_mean   %.3e   (flags translation too)' % drift_point)
print('  peak density            %.2f -> %.2f' % (n_before.max(), n_after.max()))
print('  contrast change         %.3e   (shape preservation)' % dcon)
print('  crystal slide           %+.4f xi  = %.2f%% of a' % (slide, 100*abs(slide)/a_m))
print('  Bragg amplitude change  %.3e   (droplet rearrangement)' % dA)

solver_ok = (dE < 1e-6) and (dN < 1e-9)
shape_ok = dcon < 1e-2
rigid_slide = abs(slide) > 0.05*a_m
rearranging = dA > 0.05
print()
if not solver_ok:
    cause = 'SOLVER PROBLEM -- energy or norm not conserved'
elif not shape_ok:
    cause = 'DROPLET COLLAPSE or breathing -- LHY may be too weak'
elif rigid_slide and not rearranging:
    cause = 'RIGID TRANSLATION only -- Goldstone mode, physically fine'
elif rearranging:
    cause = ('DROPLETS REARRANGING -- state is in the wrong local minimum; '
             'an optimisation problem, not a physics bug')
else:
    cause = 'STATIONARY'
print('DIAGNOSIS: %s' % cause)
stationary = (solver_ok and shape_ok and not rearranging)

if unstable and contrast > 1.0 and err < 10.0 and stationary:
    verdict = 'DROPLET CRYSTAL at the roton period, converged and stationary'
elif unstable and contrast > 1.0 and err < 10.0:
    verdict = ('DROPLET CRYSTAL at the roton period, NOT the converged '
               'ground state -- ' + cause)
elif contrast <= 1.0:
    verdict = 'NO CRYSTAL formed (relaxation too short or wrong parameters)'
else:
    verdict = 'NO CRYSTAL at these parameters'
print('VERDICT: %s' % verdict)

run.save_checkpoint(s.psi, step=24000, t=0.0)
run.save_diagnostics([{'eps_dd': eps_dd, 'l_z': l_z, 'gamma': gam,
                       'k_rot': float(k_rot), 'a_predicted': float(a_pred),
                       'lambda_measured': float(lam_m),
                       'a_triangular_measured': float(a_tri_m),
                       'a_triangular_predicted': float(a_tri_p),
                       'period_error_pct': err,
                       'optimizer': 'multistart L-BFGS-B',
                       'optimizer_nit': opt['nit'],
                       'contrast': contrast, 'mu': mu, 'residual': res,
                       'realtime_drift_pointwise': drift_point,
                       'bragg_amplitude_change': dA,
                       'contrast_change': dcon,
                       'crystal_slide_xi': slide,
                       'energy_drift': dE,
                       'diagnosis': cause,
                       'stationary': bool(stationary),
                       'verdict': verdict}])
run.finish('completed')

print()
print('STATUS OF VALIDATION RUNG V4:')
print('  DONE   : quasi-2D kernel derived, verified to 5e-15, and shown')
print('           to produce a genuine droplet crystal whose period')
print('           agrees with roton theory to about 2 percent.')
print('  NOT DONE: the crystal is not the converged ground state.')
print('           v1.5 measurements rule out solver error, droplet')
print('           collapse, and a trivial Goldstone slide.  What remains')
print('           is that the droplets rearrange: the landscape has many')
print('           nearby minima and imaginary-time gradient flow from one')
print('           seed lands in the wrong one.  The fix is a real')
print('           minimiser.  v1.6 implements exactly that: multi-start')
print('           L-BFGS-B, which lowers mu from 5.509 (24000 gradient-flow')
print('           steps) to 5.189.  The residual still sits near 5e-3')
print('           rather than 1e-8, so the landscape remains rugged and')
print('           the global minimum is not proven.  Comparing the lattice')
print('           constant to published 164Dy values -- the actual content')
print('           of V4 -- still awaits that.')

## 5. Campaign run

In [ ]:
# CELL 5 - Campaign run: driven tangle, then free decay.
#
# The v1.1 stirring protocol nucleates vortices; the v1.0 one did not.
# Two conditions must hold and v1.0 met neither: the obstacle must
# pierce the fluid (V0 >~ mu; v1.0 had 0.22 mu) and the flow must be
# fast enough to shed (Ma >~ 0.5; v1.0 had 0.35).  v1.0 therefore
# reported n_vortex = 0 for every step while pumping only sound.
import subprocess, sys

# v1.6: the campaign now defaults to the quasi2d kernel.  Running
# turbulence on the bare kernel would be turbulence in a rippled
# superfluid, not a supersolid -- which defeats the point of the project.
cmd = [sys.executable, 'campaign/template_scan.py',
       '--eps_dd', '1.8', '--seed', '0',
       '--kernel', 'quasi2d', '--l_z', '6.0',
       '--N', '128', '--L', '64', '--dt', '0.005',
       '--Ma', '1.1', '--V0_factor', '3.0', '--n_stir', '3',
       '--T_relax', '20', '--T_stir', '30', '--T_decay', '40',
       '--checkpoint_every', '1000', '--use_archive']
r = subprocess.run(cmd, cwd=DEST, capture_output=True, text=True)
print(r.stdout[-3500:])
if r.stderr.strip():
    print('STDERR:', r.stderr[-1500:])

print()
print('WHAT TO CHECK:')
print(' 1. kernel=quasi2d and "roton-unstable: True" -- otherwise the')
print('    background is not crystalline and this is not supersolid QT.')
print(' 2. vortex count climbs during stirring, then falls in free decay,')
print('    with E_i dropping while E_c rises (vortex energy -> sound).')
print(' 3. charge stays neutral: n_vortex_plus == n_vortex_minus at every')
print('    sample.  Required topologically in a periodic box.')

## 6. Archive contents

In [ ]:
# CELL 6 - Everything saved in Drive, run by run.
import os

print('=' * 62)
archive.list_runs()
print('=' * 62)

for r in archive.load_index().get('runs', []):
    d = os.path.join(archive.project_dir, r['run_id'])
    print()
    print(r['run_id'])
    print('   title   :', r.get('title', '?'))
    print('   started :', r.get('started', '?'))
    print('   status  :', r.get('status', '?'))
    if r.get('parameters'):
        print('   params  :', r['parameters'])
    for sub in ('checkpoints', 'diagnostics', 'figures', 'logs'):
        p = os.path.join(d, sub)
        if not os.path.isdir(p):
            continue
        for f in sorted(os.listdir(p)):
            kb = os.path.getsize(os.path.join(p, f)) / 1024.0
            print('     %s/%s  (%.1f KB)' % (sub, f, kb))

print()
# Print the ACTUAL location.  This line used to be a hardcoded string
# claiming 'MyDrive/...' even when output had gone to ephemeral disk.
print('Actual output location:', archive.project_dir)
print('Persistent (on Google Drive):', archive.persistent)
if not archive.persistent:
    print('  !! These files are EPHEMERAL and will be lost.')